# 02 - Extração dos PDFs

Esta etapa lê o manifesto da Etapa 1, seleciona apenas conteúdos únicos e extrai o texto página a página com PyMuPDF.

O notebook **não executa OCR**. Ele apenas identifica artigos possivelmente escaneados ou com extração insuficiente para revisão posterior.

In [1]:
import pandas as pd

from wonca_rag.paths import PATHS
from wonca_rag.ingestion.pdf_extractor import (
    choose_unique_articles,
    extract_unique_articles,
    save_extraction_report,
    summarize_extraction,
)

MANIFEST_PATH = PATHS.data_root / 'metadata' / 'articles_manifest.parquet'
PDF_ROOT = PATHS.data_root / 'raw' / 'pdf'
PROCESSED_ROOT = PATHS.data_root / 'processed'
CHECKPOINTS_ROOT = PATHS.outputs_root / 'checkpoints' / 'pdf_extraction'
REPORT_DIR = PATHS.outputs_root / 'logs'

print('MANIFEST_PATH:', MANIFEST_PATH)
print('PDF_ROOT:', PDF_ROOT)
print('PROCESSED_ROOT:', PROCESSED_ROOT)

MANIFEST_PATH: C:\Github\wonca-rag\data\metadata\articles_manifest.parquet
PDF_ROOT: C:\Github\wonca-rag\data\raw\pdf
PROCESSED_ROOT: C:\Github\wonca-rag\data\processed


## 1. Carregar o manifesto

In [2]:
manifest_df = pd.read_parquet(MANIFEST_PATH)
print('Arquivos físicos no manifesto:', len(manifest_df))
print('Conteúdos únicos por SHA-256:', manifest_df['sha256'].nunique())
display(manifest_df.head())

Arquivos físicos no manifesto: 233
Conteúdos únicos por SHA-256: 215


,article_id,filename,relative_path,source_folder,file_size_bytes,sha256,is_duplicate_content,duplicate_group,inferred_first_author,publication_year,publication_date,temporal_group,year_inferred_from_filename,needs_metadata_review
0,ART_E3A2395BE56C,Starfield1967.pdf,Starfield1967.pdf,,339081,e3a2395be56c7e7b3fef9df7fb5524dd7e5236d14ae32f...,False,NaN,Starfield,1967,NaT,1967,True,False
1,ART_8FCDDCD40C31,Starfield1968.pdf,Starfield1968.pdf,,916897,8fcddcd40c316d5087887a45e990d13085ced7a43dedc1...,False,NaN,Starfield,1968,NaT,1968,True,False
2,ART_C31E1FAB9674,Starfield1968b.pdf,Starfield1968b.pdf,,517352,c31e1fab9674f3fb6f7dd939b18f0f90417a761e0a7f03...,False,NaN,Starfield,1968,NaT,1968,True,False
3,ART_EB1086AB18B2,densen1968.pdf,densen1968.pdf,,618306,eb1086ab18b2c10481e59278bbf3a30a520f454443ec46...,False,NaN,densen,1968,NaT,1968,True,False
4,ART_8063B5B4E555,Starfield1969.pdf,Starfield1969.pdf,,723755,8063b5b4e555ef0dc5ff8e9076a1aca201bb7901d6ebe4...,False,NaN,Starfield,1969,NaT,1969,True,False


## 2. Conferir quais conteúdos serão processados

In [3]:
unique_df = choose_unique_articles(manifest_df)
print('Artigos únicos para extração:', len(unique_df))
display(unique_df[['article_id', 'filename', 'relative_path', 'publication_year']].head(20))

Artigos únicos para extração: 215


,article_id,filename,relative_path,publication_year
0,ART_8F765D144499,Abramson2005.pdf,Abramson2005.pdf,2005
1,ART_2179244B822A,Allen1989.pdf,Allen1989.pdf,1989
2,ART_404FD0BBD2B8,Almeida2001.pdf,Almeida2001.pdf,2001
3,ART_E58422CEE3D5,Alonso2008.pdf,Alonso2008.pdf,2008
4,ART_CC8D0D4F8DEE,Barker1989.pdf,Artigos sem resumo/Barker1989.pdf,1989
5,ART_C51FA4EF984E,Starfield1971b (1).pdf,Artigos sem resumo/Starfield1971b (1).pdf,1971
6,ART_B36F04112348,Starfield1971c.pdf,Artigos sem resumo/Starfield1971c.pdf,1971
7,ART_0080057343D5,Starfield1976b.pdf,Artigos sem resumo/Starfield1976b.pdf,1976
8,ART_5235CCA4CEF6,Starfield1981d.pdf,Artigos sem resumo/Starfield1981d.pdf,1981
9,ART_27019C5BC11E,Starfield1982c.pdf,Artigos sem resumo/Starfield1982c.pdf,1982


## 3. Executar extração

A opção `resume=True` faz com que artigos já extraídos com o mesmo SHA-256 e a mesma versão do extrator sejam ignorados.

In [4]:
report_df = extract_unique_articles(
    manifest_df=manifest_df,
    pdf_root=PDF_ROOT,
    processed_root=PROCESSED_ROOT,
    checkpoints_root=CHECKPOINTS_ROOT,
    resume=True,
)

display(report_df.head())

Extracting PDFs:   0%|          | 0/215 [00:00<?, ?it/s]

,article_id,sha256,filename,relative_path,pdf_path,article_json,checkpoint_json,status,error_type,error_message,n_pages,total_characters,mean_characters_per_page,empty_pages,empty_page_ratio,likely_scanned
0,ART_8F765D144499,8f765d14449960d97572b150550a93fef5cac5c22f0817...,Abramson2005.pdf,Abramson2005.pdf,C:\Github\wonca-rag\data\raw\pdf\Abramson2005.pdf,C:\Github\wonca-rag\data\processed\ART_8F765D1...,C:\Github\wonca-rag\outputs\checkpoints\pdf_ex...,SKIPPED_CURRENT,None,None,5,22562,4512.400000,0,0.0,False
1,ART_2179244B822A,2179244b822a8bc3cee601b16b87678e023f070a5d61b6...,Allen1989.pdf,Allen1989.pdf,C:\Github\wonca-rag\data\raw\pdf\Allen1989.pdf,C:\Github\wonca-rag\data\processed\ART_2179244...,C:\Github\wonca-rag\outputs\checkpoints\pdf_ex...,SKIPPED_CURRENT,None,None,4,24708,6177.000000,0,0.0,False
2,ART_404FD0BBD2B8,404fd0bbd2b8fdf55aaf93fb679af45b3115ad4754624d...,Almeida2001.pdf,Almeida2001.pdf,C:\Github\wonca-rag\data\raw\pdf\Almeida2001.pdf,C:\Github\wonca-rag\data\processed\ART_404FD0B...,C:\Github\wonca-rag\outputs\checkpoints\pdf_ex...,SKIPPED_CURRENT,None,None,6,39539,6589.833333,0,0.0,False
3,ART_E58422CEE3D5,e58422cee3d5e0eeb543b2918fe041e7cc1e8d4ecd37b6...,Alonso2008.pdf,Alonso2008.pdf,C:\Github\wonca-rag\data\raw\pdf\Alonso2008.pdf,C:\Github\wonca-rag\data\processed\ART_E58422C...,C:\Github\wonca-rag\outputs\checkpoints\pdf_ex...,SKIPPED_CURRENT,None,None,11,41540,3776.363636,0,0.0,False
4,ART_CC8D0D4F8DEE,cc8d0d4f8deed88fc7a21478e72b25555c65c120e35a8a...,Barker1989.pdf,Artigos sem resumo/Barker1989.pdf,C:\Github\wonca-rag\data\raw\pdf\Artigos sem r...,C:\Github\wonca-rag\data\processed\ART_CC8D0D4...,C:\Github\wonca-rag\outputs\checkpoints\pdf_ex...,SKIPPED_CURRENT,None,None,5,0,0.000000,5,1.0,True


## 4. Resumo de qualidade

In [5]:
summary = summarize_extraction(report_df)
display(pd.DataFrame(summary.items(), columns=['metric', 'value']))

,metric,value
0,n_unique_articles,215
1,n_done,0
2,n_skipped_current,215
3,n_error,0
4,n_likely_scanned,11
5,n_zero_text,11


## 5. Artigos com erro

In [6]:
errors_df = report_df.loc[report_df['status'] == 'ERROR'].copy()
print('Erros:', len(errors_df))
display(errors_df[['article_id', 'filename', 'error_type', 'error_message']].head(100))

Erros: 0


,article_id,filename,error_type,error_message


## 6. PDFs possivelmente escaneados ou com pouco texto

In [7]:
scanned_df = report_df.loc[
    report_df['likely_scanned'].fillna(False)
].copy()

print('Possivelmente escaneados:', len(scanned_df))
display(
    scanned_df[[
        'article_id',
        'filename',
        'n_pages',
        'total_characters',
        'mean_characters_per_page',
        'empty_pages',
        'empty_page_ratio',
    ]].sort_values('mean_characters_per_page').head(100)
)

Possivelmente escaneados: 11


,article_id,filename,n_pages,total_characters,mean_characters_per_page,empty_pages,empty_page_ratio
4,ART_CC8D0D4F8DEE,Barker1989.pdf,5,0,0.0,5,1.0
7,ART_0080057343D5,Starfield1976b.pdf,2,0,0.0,2,1.0
13,ART_87286A208679,Starfield1993.pdf,11,0,0.0,11,1.0
29,ART_00098517AA25,Bice1973.pdf,10,0,0.0,10,1.0
33,ART_4C1CCB6ED34C,Carr1993.pdf,5,0,0.0,5,1.0
41,ART_331895B7FCEB,Diaz1986.pdf,9,0,0.0,9,1.0
122,ART_757497A5F654,Maguire1998.pdf,10,0,0.0,10,1.0
153,ART_8AC4523EBB7F,Reid1996.pdf,7,0,0.0,7,1.0
154,ART_007D4934BC69,Riley1993.pdf,17,0,0.0,17,1.0
168,ART_34868B1241D9,Schor1987.pdf,11,0,0.0,11,1.0


## 7. Artigos com zero caracteres extraídos

In [8]:
zero_text_df = report_df.loc[
    pd.to_numeric(report_df['total_characters'], errors='coerce').fillna(-1).eq(0)
].copy()

print('Zero texto:', len(zero_text_df))
display(zero_text_df[['article_id', 'filename', 'n_pages', 'total_characters']].head(100))

Zero texto: 11


,article_id,filename,n_pages,total_characters
4,ART_CC8D0D4F8DEE,Barker1989.pdf,5,0
7,ART_0080057343D5,Starfield1976b.pdf,2,0
13,ART_87286A208679,Starfield1993.pdf,11,0
29,ART_00098517AA25,Bice1973.pdf,10,0
33,ART_4C1CCB6ED34C,Carr1993.pdf,5,0
41,ART_331895B7FCEB,Diaz1986.pdf,9,0
122,ART_757497A5F654,Maguire1998.pdf,10,0
153,ART_8AC4523EBB7F,Reid1996.pdf,7,0
154,ART_007D4934BC69,Riley1993.pdf,17,0
168,ART_34868B1241D9,Schor1987.pdf,11,0


## 8. Salvar relatório da extração

In [9]:
report_paths = save_extraction_report(
    report_df=report_df,
    output_dir=REPORT_DIR,
)

print('Relatórios salvos:')
for name, path in report_paths.items():
    print('-', name, ':', path)

Relatórios salvos:
- parquet : C:\Github\wonca-rag\outputs\logs\pdf_extraction_report.parquet
- xlsx : C:\Github\wonca-rag\outputs\logs\pdf_extraction_report.xlsx
- csv : C:\Github\wonca-rag\outputs\logs\pdf_extraction_report.csv


## Resultado esperado

- Um JSON por artigo único em `data/processed/`
- Um checkpoint por artigo em `outputs/checkpoints/pdf_extraction/`
- Relatórios em `outputs/logs/`

Todos esses artefatos são gerados localmente e não devem ser enviados ao Git.